# Pipeline de ingesta: Kaggle → HDFS → MinIO

## Objetivo
Construir un mini pipeline de datos que simula el flujo real de ingestión en una empresa:

```
Kaggle (fuente externa)
       ↓  descarga y lectura con PyArrow
HDFS   (zona RAW — datos tal cual llegaron, en Parquet)
       ↓  lectura + transformación con PyArrow
MinIO  (zona GOLD — datos limpios y enriquecidos, particionados)
```

### Dataset elegido
**Supermarket Sales** (`aungpyaeap/supermarket-sales`)
Ventas de un supermercado con columnas de ciudad, línea de producto, cantidad, precio, fecha, etc.
Es pequeño (~1000 filas), limpio y perfecto para practicar transformaciones reales.

### Lo que practicarás
- Autenticación con la API de Kaggle
- Lectura eficiente con **PyArrow** (sin pasar por Pandas)
- Escritura y lectura en **HDFS** vía`pyarrow.fs.HadoopFileSystem`
- Transformaciones con **PyArrow compute** (sin Pandas)
- Escritura particionada en **MinIO/S3** con`pyarrow.parquet.write_to_dataset`

> **Antes de empezar:** necesitas una cuenta gratuita en [kaggle.com](https://www.kaggle.com) y tu fichero`kaggle.json` con las credenciales de la API.

## Paso 0 — Credenciales y librería
---

In [61]:
# Librerías necesarias (todas ya disponibles en el entorno)
import os
import io
import json
import pathlib

import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import pyarrow.csv as pa_csv
import pandas as pd          # solo para display() cómodo al final
import boto3
import s3fs
from hdfs import InsecureClient

print(f"PyArrow  : {pa.__version__}")
print(f"Pandas   : {pd.__version__}")
print("Librerías listas.")

PyArrow  : 13.0.0
Pandas   : 2.0.3
Librerías listas.


In [62]:
# ── Credenciales Kaggle ───────────────────────────────────────────────────────
# La librería opendatasets te pedirá tu usuario y API key de Kaggle
# de forma interactiva al descargar por primera vez.
#
# Para obtener tus credenciales:
#   1. Ve a kaggle.com > tu perfil > Settings > API > Create New Token
#   2. Se descargará un archivo kaggle.json con tu usuario y key
#   3. Cuando opendatasets te pregunte, introduce esos datos
#
# Alternativa: coloca tu kaggle.json en /root/.kaggle/kaggle.json
# y opendatasets lo usará automáticamente sin preguntar.

!pip install -q opendatasets
import opendatasets as od
print("opendatasets instalado y listo.")


opendatasets instalado y listo.


## Paso 1 — Descargar el dataset desde Kaggle
---

In [63]:
# ── Descargar dataset desde Kaggle con opendatasets ─dataset_url = "https://www.kaggle.com/datasets/faresashraf1001/supermarket-sales"

od.download("https://www.kaggle.com/datasets/faresashraf1001/supermarket-sales")

# opendatasets descarga en una carpeta con el nombre del dataset
DOWNLOAD_DIR = pathlib.Path("./supermarket-sales")

ficheros = list(DOWNLOAD_DIR.glob("*.csv"))
print(f"\nFicheros descargados: {[f.name for f in ficheros]}")


Skipping, found downloaded files in "./supermarket-sales" (use force=True to force download)

Ficheros descargados: ['SuperMarket Analysis.csv']


In [64]:
# ── Leer el CSV con PyArrow (no con Pandas) ───────────────────────────────────
# PyArrow puede inferir tipos automáticamente y es mucho más eficiente en memoria
CSV_PATH = DOWNLOAD_DIR / "SuperMarket Analysis.csv"

# ConvertOptions permite tipar columnas concretas en la lectura
convert_opts = pa_csv.ConvertOptions(
    column_types={
        "Unit price" : pa.float32(),
        "Quantity"   : pa.int32(),
        "Tax 5%"     : pa.float32(),
        "Total"      : pa.float32(),
        "cogs"       : pa.float32(),
        "Rating"     : pa.float32(),
    }
)

tabla_raw = pa_csv.read_csv(str(CSV_PATH), convert_options=convert_opts)

print(f"Filas: {tabla_raw.num_rows}  |  Columnas: {tabla_raw.num_columns}")
print("\nEsquema PyArrow:")
print(tabla_raw.schema)

Filas: 1000  |  Columnas: 17

Esquema PyArrow:
Invoice ID: string
Branch: string
City: string
Customer type: string
Gender: string
Product line: string
Unit price: float
Quantity: int32
Tax 5%: float
Sales: double
Date: string
Time: string
Payment: string
cogs: float
gross margin percentage: double
gross income: double
Rating: float


In [65]:
# Vista rápida de las primeras filas (convertimos solo para el display)
tabla_raw.slice(0, 5).to_pandas()

,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Sales,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating
0,750-67-8428,Alex,Yangon,Member,Female,Health and beauty,74.690002,7,26.1415,548.9715,1/5/2019,1:08:00 PM,Ewallet,522.830017,4.761905,26.1415,9.1
1,226-31-3081,Giza,Naypyitaw,Normal,Female,Electronic accessories,15.280000,5,3.8200,80.2200,3/8/2019,10:29:00 AM,Cash,76.400002,4.761905,3.8200,9.6
2,631-41-3108,Alex,Yangon,Normal,Female,Home and lifestyle,46.330002,7,16.2155,340.5255,3/3/2019,1:23:00 PM,Credit card,324.309998,4.761905,16.2155,7.4
3,123-19-1176,Alex,Yangon,Member,Female,Health and beauty,58.220001,8,23.2880,489.0480,1/27/2019,8:33:00 PM,Ewallet,465.760010,4.761905,23.2880,8.4
4,373-73-7910,Alex,Yangon,Member,Female,Sports and travel,86.309998,7,30.2085,634.3785,2/8/2019,10:37:00 AM,Ewallet,604.169983,4.761905,30.2085,5.3



## Paso 2 — Guardar los datos RAW en HDFS (formato Parquet)
---
La zona RAW de un Data Lake almacena los datos exactamente como llegaron de la fuente, sin transformar.
Usamos Parquet porque:
- Comprime mucho mejor que CSV
- Guarda el esquema de tipos (no hay ambigüedad al leer)
- Spark y otras herramientas lo leen de forma columnar (mucho más rápido para analítica)

In [66]:
# ── Conexión a HDFS ───────────────────────────────────────────────────────────
hdfs = InsecureClient("http://namenode:9870", user="jovyan")

HDFS_RAW_PATH = "/data/raw/supermarket_sales.parquet"

# Serializar la tabla Arrow a bytes Parquet en memoria
buffer = io.BytesIO()
pq.write_table(
    tabla_raw,
    buffer,
    compression="snappy",   # Snappy: buen equilibrio velocidad/compresión
)
buffer.seek(0)

# Subir el stream directamente a HDFS
hdfs.makedirs("/data/raw", permission=755)
with hdfs.write(HDFS_RAW_PATH, overwrite=True) as writer:
    writer.write(buffer.read())

# Verificar que llegó
info = hdfs.status(HDFS_RAW_PATH)
print(f"Fichero guardado en HDFS: {HDFS_RAW_PATH}")
print(f"Tamaño en HDFS: {info['length']:,} bytes  ({info['length']/1024:.1f} KB)")

Fichero guardado en HDFS: /data/raw/supermarket_sales.parquet
Tamaño en HDFS: 54,994 bytes  (53.7 KB)


In [67]:
# ── Verificación: leer de vuelta desde HDFS ───────────────────────────────────
with hdfs.read(HDFS_RAW_PATH) as f:
    tabla_desde_hdfs = pq.read_table(io.BytesIO(f.read()))

print(f"Leído desde HDFS: {tabla_desde_hdfs.num_rows} filas x {tabla_desde_hdfs.num_columns} columnas")
print("Esquema recuperado correctamente:", tabla_desde_hdfs.schema.equals(tabla_raw.schema))

Leído desde HDFS: 1000 filas x 17 columnas
Esquema recuperado correctamente: True


## Paso 3 — Transformaciones con PyArrow compute
---
Aplicamos las transformaciones propias de la zona GOLD:
1. Parsear la columna`Date` a tipo`date32`
2. Calcular`gross_income_eur` (simulando conversión a euros, factor 0.92)
3. Añadir columna`rating_category` (Alto / Medio / Bajo)
4. Seleccionar solo las columnas relevantes para análisis
5. Renombrar columnas al estándar snake_case

In [68]:
# Trabajamos desde la tabla leída de HDFS (simula que el proceso de transformación
# corre en un nodo distinto al que hizo la ingestión)
t = tabla_desde_hdfs

# ── 3.1 Parsear fecha ─────────────────────────────────────────────────────────
# La columna 'Date' llega como string "1/5/2019"
fechas_str   = t.column("Date")                             # Arrow StringArray
fechas_ts    = pc.strptime(fechas_str, format="%m/%d/%Y", unit="s")  # timestamp[s]
fechas_date  = pc.cast(fechas_ts, pa.date32())              # date32

print("Muestra de fechas parseadas:", fechas_date[:3].to_pylist())

Muestra de fechas parseadas: [datetime.date(2019, 1, 5), datetime.date(2019, 3, 8), datetime.date(2019, 3, 3)]


In [69]:
#
print(t.column_names)


['Invoice ID', 'Branch', 'City', 'Customer type', 'Gender', 'Product line', 'Unit price', 'Quantity', 'Tax 5%', 'Sales', 'Date', 'Time', 'Payment', 'cogs', 'gross margin percentage', 'gross income', 'Rating']


In [70]:
# ── 3.2 Conversión de moneda ──────────────────────────────────────────────────
EUR_FACTOR = 0.92
total_eur = pc.multiply(t.column("Sales"), pa.scalar(EUR_FACTOR, pa.float32()))
gross_eur = pc.multiply(t.column("gross income"), pa.scalar(EUR_FACTOR, pa.float32()))

# ── 3.3 Categoría de rating ───────────────────────────────────────────────────
# Alto: rating >= 8  |  Medio: 6 <= rating < 8  |  Bajo: < 6
rating = t.column("Rating")
rating_cat = pc.case_when(
    pc.make_struct(
        pc.greater_equal(rating, 8.0),
        pc.and_(
            pc.greater_equal(rating, 6.0),
            pc.less(rating, 8.0)
        )
    ),
    pa.array(["Alto"]   * t.num_rows, pa.string()),
    pa.array(["Medio"]  * t.num_rows, pa.string()),
    pa.array(["Bajo"]   * t.num_rows, pa.string()),   # else
)

print("Distribución de rating_category:")
cats, counts = pa.compute.value_counts(rating_cat).flatten()
for c, n in zip(cats.to_pylist(), counts.to_pylist()):
    print(f"  {c}: {n}")

Distribución de rating_category:
  Alto: 329
  Medio: 356
  Bajo: 315


In [71]:
# ── 3.4 Construir la tabla GOLD ───────────────────────────────────────────────
tabla_gold = pa.table({
    "invoice_id"      : t.column("Invoice ID"),
    "branch"          : t.column("Branch"),
    "city"            : t.column("City"),
    "customer_type"   : t.column("Customer type"),
    "gender"          : t.column("Gender"),
    "product_line"    : t.column("Product line"),
    "unit_price"      : t.column("Unit price"),
    "quantity"        : t.column("Quantity"),
    "total_usd"       : t.column("Sales"),
    "total_eur"       : total_eur,
    "gross_income_eur": gross_eur,
    "date"            : fechas_date,
    "payment"         : t.column("Payment"),
    "rating"          : t.column("Rating"),
    "rating_category" : rating_cat,
})

print(f"Tabla GOLD: {tabla_gold.num_rows} filas x {tabla_gold.num_columns} columnas")
print("\nEsquema GOLD:")
print(tabla_gold.schema)

Tabla GOLD: 1000 filas x 15 columnas

Esquema GOLD:
invoice_id: string
branch: string
city: string
customer_type: string
gender: string
product_line: string
unit_price: float
quantity: int32
total_usd: double
total_eur: double
gross_income_eur: double
date: date32[day]
payment: string
rating: float
rating_category: string


In [72]:
# Vista de las primeras filas
tabla_gold.slice(0, 5).to_pandas()

,invoice_id,branch,city,customer_type,gender,product_line,unit_price,quantity,total_usd,total_eur,gross_income_eur,date,payment,rating,rating_category
0,750-67-8428,Alex,Yangon,Member,Female,Health and beauty,74.690002,7,548.9715,505.053789,24.050180,2019-01-05,Ewallet,9.1,Alto
1,226-31-3081,Giza,Naypyitaw,Normal,Female,Electronic accessories,15.280000,5,80.2200,73.802401,3.514400,2019-03-08,Cash,9.6,Alto
2,631-41-3108,Alex,Yangon,Normal,Female,Home and lifestyle,46.330002,7,340.5255,313.283466,14.918260,2019-03-03,Credit card,7.4,Medio
3,123-19-1176,Alex,Yangon,Member,Female,Health and beauty,58.220001,8,489.0480,449.924168,21.424960,2019-01-27,Ewallet,8.4,Alto
4,373-73-7910,Alex,Yangon,Member,Female,Sports and travel,86.309998,7,634.3785,583.628231,27.791821,2019-02-08,Ewallet,5.3,Bajo


## Paso 4 — Escribir los datos GOLD en MinIO (particionados)
---

En la zona GOLD escribimos los datos **particionados por`city`**.
Esto significa que MinIO tendrá esta estructura:
```
s3://datalake/gold/supermarket/
    city=Mandalay/
        part-0.parquet
    city=Naypyitaw/
        part-0.parquet
    city=Yangon/
        part-0.parquet
```
Cuando Spark o DuckDB lean esta carpeta, filtrar por ciudad es gratuito (no lee las otras particiones).

In [73]:
# ── Conexión a MinIO ──────────────────────────────────────────────────────────
MINIO_ENDPOINT = "http://minio:9000"
ACCESS_KEY     = "admin"
SECRET_KEY     = "adminadmin"

s3_client = boto3.client(
    "s3",
    endpoint_url          = MINIO_ENDPOINT,
    aws_access_key_id     = ACCESS_KEY,
    aws_secret_access_key = SECRET_KEY,
)

# Crear el bucket 'datalake' si no existe
from botocore.exceptions import ClientError
try:
    s3_client.create_bucket(Bucket="datalake")
    print("Bucket 'datalake' creado.")
except ClientError as e:
    print(f"Bucket ya existe o aviso: {e.response['Error']['Code']}")

Bucket ya existe o aviso: BucketAlreadyOwnedByYou


In [74]:
# ── Escribir dataset particionado directamente en MinIO usando s3fs + PyArrow ──
fs_s3 = s3fs.S3FileSystem(
    key               = ACCESS_KEY,
    secret            = SECRET_KEY,
    client_kwargs     = {"endpoint_url": MINIO_ENDPOINT},
)

S3_GOLD_PATH = "datalake/gold/supermarket"

pq.write_to_dataset(
    tabla_gold,
    root_path         = S3_GOLD_PATH,
    partition_cols    = ["city"],          # Una sub-carpeta por ciudad
    filesystem        = fs_s3,
    compression       = "snappy",
    existing_data_behavior = "overwrite_or_ignore",
)

print(f"Dataset GOLD escrito en s3://{S3_GOLD_PATH}")

Dataset GOLD escrito en s3://datalake/gold/supermarket


In [75]:
# ── Verificar particiones creadas en MinIO ────────────────────────────────────
respuesta = s3_client.list_objects_v2(Bucket="datalake", Prefix="gold/supermarket/")

print("Objetos en la zona GOLD:")
for obj in respuesta.get("Contents", []):
    print(f"  s3://datalake/{obj['Key']}  ({obj['Size']:,} bytes)")

Objetos en la zona GOLD:
  s3://datalake/gold/supermarket/city=Mandalay/734d63166e934117aa124400fdccd6f1-0.parquet  (21,248 bytes)
  s3://datalake/gold/supermarket/city=Naypyitaw/734d63166e934117aa124400fdccd6f1-0.parquet  (21,050 bytes)
  s3://datalake/gold/supermarket/city=Yangon/734d63166e934117aa124400fdccd6f1-0.parquet  (21,596 bytes)



## Paso 5 — Lectura final y validación
---
Leer el dataset GOLD de vuelta desde MinIO para verificar que todo está correcto.
Aprovechamos para demostrar el **filtrado por partición** (solo cargamos una ciudad).

In [76]:
# ── Leer el dataset completo desde MinIO ──────────────────────────────────────
dataset_gold = pq.ParquetDataset(
    S3_GOLD_PATH,
    filesystem = fs_s3,
)
tabla_leida = dataset_gold.read()

print(f"Total filas recuperadas: {tabla_leida.num_rows}")
print(f"Ciudades presentes: {pc.unique(tabla_leida.column('city')).to_pylist()}")

Total filas recuperadas: 1000
Ciudades presentes: ['Mandalay', 'Naypyitaw', 'Yangon']


In [77]:
# ── Filtrado por partición: solo Yangon ───────────────────────────────────────
# PyArrow usará partition pruning: solo leerá city=Yangon/, ignorará las otras
tabla_yangon = pq.ParquetDataset(
    S3_GOLD_PATH,
    filesystem = fs_s3,
    filters    = [("city", "=", "Yangon")],
).read()

print(f"Filas solo de Yangon: {tabla_yangon.num_rows}")
tabla_yangon.slice(0, 5).to_pandas()

Filas solo de Yangon: 340


,invoice_id,branch,customer_type,gender,product_line,unit_price,quantity,total_usd,total_eur,gross_income_eur,date,payment,rating,rating_category,city
0,750-67-8428,Alex,Member,Female,Health and beauty,74.690002,7,548.9715,505.053789,24.050180,2019-01-05,Ewallet,9.1,Alto,Yangon
1,631-41-3108,Alex,Normal,Female,Home and lifestyle,46.330002,7,340.5255,313.283466,14.918260,2019-03-03,Credit card,7.4,Medio,Yangon
2,123-19-1176,Alex,Member,Female,Health and beauty,58.220001,8,489.0480,449.924168,21.424960,2019-01-27,Ewallet,8.4,Alto,Yangon
3,373-73-7910,Alex,Member,Female,Sports and travel,86.309998,7,634.3785,583.628231,27.791821,2019-02-08,Ewallet,5.3,Bajo,Yangon
4,355-53-5943,Alex,Member,Female,Electronic accessories,68.839996,6,433.6920,398.996647,18.999840,2019-02-25,Ewallet,5.8,Bajo,Yangon


In [78]:
# ── Agregación final: ingresos brutos totales por ciudad y línea de producto ──
import pyarrow.dataset as ds

# Usamos groupby de PyArrow (disponible desde PyArrow 13)
resumen = (
    tabla_leida
    .group_by(["city", "product_line"])
    .aggregate([
        ("gross_income_eur", "sum"),
        ("quantity",         "sum"),
        ("rating",           "mean"),
    ])
    .sort_by([("gross_income_eur_sum", "descending")])
)

print("TOP combinaciones ciudad × línea de producto por ingresos:")
resumen.to_pandas()

TOP combinaciones ciudad × línea de producto por ingresos:


,city,product_line,gross_income_eur_sum,quantity_sum,rating_mean
0,Naypyitaw,Food and beverages,1041.214619,369,7.080303
1,Yangon,Home and lifestyle,982.086678,371,6.930769
2,Naypyitaw,Fashion accessories,944.536417,342,7.440000
3,Mandalay,Sports and travel,875.673496,322,6.509677
4,Mandalay,Health and beauty,875.343216,320,7.100000
5,Yangon,Sports and travel,848.708755,333,7.257627
6,Naypyitaw,Electronic accessories,831.021755,333,6.747273
7,Yangon,Electronic accessories,802.464035,322,6.911667
8,Mandalay,Home and lifestyle,768.820554,295,6.516000
9,Yangon,Food and beverages,751.907274,313,7.253448



## Resumen del pipeline completado
---
```
Kaggle API
  └─► CSV descargado en /tmp/kaggle_supermarket/
        └─► PyArrow CSV reader (tipado explícito)
              └─► HDFS /data/raw/supermarket_sales.parquet  (compresión Snappy)
                    └─► Transformaciones PyArrow compute
                          │  · Parseo de fechas (strptime → date32)
                          │  · Conversión USD → EUR
                          │  · Categorización de rating
                          │  · Renombrado snake_case
                          └─► MinIO s3://datalake/gold/supermarket/
                                city=Mandalay/  part-0.parquet
                                city=Naypyitaw/ part-0.parquet
                                city=Yangon/    part-0.parquet
```

### Puntos clave
| Concepto | Dónde lo hemos aplicado |
|---|---|
| **Zona RAW vs GOLD** | HDFS = RAW (datos originales), MinIO = GOLD (transformados) |
| **PyArrow sin Pandas** | Lectura CSV, transformaciones con`pc.*`, escritura Parquet |
| **Particionado** |`write_to_dataset` con`partition_cols=["city"]` |
| **Partition pruning** |`ParquetDataset` con`filters` solo lee las particiones necesarias |
| **Compresión Snappy** | Mejor ratio velocidad/tamaño para datos analíticos |

---
### Ejercicio propuesto
Añade una segunda partición por`payment` (además de`city`) y comprueba cuántos ficheros genera MinIO.
Luego lee solo las ventas de Yangon pagadas con`Cash`.

In [79]:
# ── Escribir dataset particionado directamente en MinIO usando s3fs + PyArrow ──
fs_s3 = s3fs.S3FileSystem(
    key               = ACCESS_KEY,
    secret            = SECRET_KEY,
    client_kwargs     = {"endpoint_url": MINIO_ENDPOINT},
)

S3_GOLD_PATH = "datalake/gold/supermarket"

pq.write_to_dataset(
    tabla_gold,
    root_path         = S3_GOLD_PATH,
    partition_cols    = ["payment"],          # Una sub-carpeta por ciudad
    filesystem        = fs_s3,
    compression       = "snappy",
    use_dictionary=False,
    existing_data_behavior = "overwrite_or_ignore",
)

print(f"Dataset GOLD particinando por pyment escrito en s3://{S3_GOLD_PATH}")

Dataset GOLD particinando por pyment escrito en s3://datalake/gold/supermarket


In [80]:
# ── 1. Borrar la estructura antigua ───────────────────────────────────────────
# ¿Por qué borramos? 
# MinIO guarda los datos particionados en carpetas físicas. Si cambiamos la 
# regla de partición (añadir 'payment'), los archivos nuevos chocarán con los 
# archivos viejos que tenían 'payment' como texto por dentro. PyArrow no sabe 
# mezclar ambas estructuras, por lo que debemos limpiar el "lienzo" primero.
print(f"Borrando datos antiguos en el bucket 'datalake', prefijo 'gold/supermarket/'...")

prefijo = "gold/supermarket/"
respuesta = s3_client.list_objects_v2(Bucket="datalake", Prefix=prefijo)

if "Contents" in respuesta:
    for obj in respuesta["Contents"]:
        # Borramos archivo por archivo (DeleteObject singular no exige MD5)
        s3_client.delete_object(Bucket="datalake", Key=obj["Key"])
    print("Datos antiguos borrados.\n")
else:
    print("La carpeta ya estaba vacía o no existía.\n")
# ── 2. Escribir el dataset con doble partición ────────────────────────────────
print("Escribiendo el dataset particionado por 'payment' y 'city'...")
pq.write_to_dataset(
    tabla_gold,
    root_path         = S3_GOLD_PATH,
    partition_cols    = ["payment", "city"],  # Particionamos por pago y luego ciudad
    filesystem        = fs_s3,
    compression       = "snappy",
)
print(f"Dataset GOLD escrito con éxito en s3://{S3_GOLD_PATH}\n")

# ── 3. Leer de vuelta y verificar los pagos ───────────────────────────────────
print("Leyendo el dataset recién creado...")
dataset_gold = pq.ParquetDataset(
    S3_GOLD_PATH,
    filesystem = fs_s3,
)
tabla_leida = dataset_gold.read()

# Mostramos las filas y los pagos únicos, tal como pedías
print(f"Total filas recuperadas: {tabla_leida.num_rows}")
print(f"Pagos presentes: {pc.unique(tabla_leida.column('payment')).to_pylist()}")

Borrando datos antiguos en el bucket 'datalake', prefijo 'gold/supermarket/'...
Datos antiguos borrados.

Escribiendo el dataset particionado por 'payment' y 'city'...
Dataset GOLD escrito con éxito en s3://datalake/gold/supermarket

Leyendo el dataset recién creado...
Total filas recuperadas: 1000
Pagos presentes: ['Cash', 'Credit card', 'Ewallet']


Luego lee solo las ventas de Yangon pagadas con Cash.

In [81]:
tabla_yangon = pq.ParquetDataset(
    S3_GOLD_PATH,
    filesystem = fs_s3,
    filters    = [("city", "=", "Yangon"),("payment", "=", "Cash")],
).read()

print(f"Filas solo de Yangon: {tabla_yangon.num_rows}")
tabla_yangon.slice(0, 5).to_pandas()

Filas solo de Yangon: 110


,invoice_id,branch,customer_type,gender,product_line,unit_price,quantity,total_usd,total_eur,gross_income_eur,date,rating,rating_category,payment,city
0,829-34-3910,Alex,Member,Female,Health and beauty,71.379997,10,749.4900,689.530813,32.834801,2019-03-29,5.7,Bajo,Cash,Yangon
1,848-62-7243,Alex,Member,Female,Health and beauty,24.889999,9,235.2105,216.393664,10.304460,2019-03-15,7.4,Medio,Cash,Yangon
2,162-48-8011,Alex,Member,Female,Food and beverages,44.590000,5,234.0975,215.369704,10.255700,2019-02-10,8.5,Alto,Cash,Yangon
3,106-35-6779,Alex,Member,Female,Home and lifestyle,44.340000,2,93.1140,85.664882,4.079280,2019-03-27,5.8,Bajo,Cash,Yangon
4,817-48-8732,Alex,Member,Female,Home and lifestyle,72.349998,10,759.6750,698.901013,33.281001,2019-01-20,5.4,Bajo,Cash,Yangon
